# Systematics: GENIE

GENIE systematics are handled differently for uncertainty on the event rates vs. uncertainty on the xsec measurement

For the latter, we only consider the impact on response for the signal channel

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
from os import path, makedirs
from datetime import datetime

import sys
sys.path.append('../../')
from analysis_village.numucc_1p0pi.variable_configs import VariableConfig
from analysis_village.numucc_1p0pi.utils import *
from pyanalib.split_df_helpers import *
from pyanalib.covariance import *

# turn off PerformanceWarning 
# triggered by mismatched column levels
import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

In [ ]:
save_result = False
save_fig = save_result

save_fig_base_dir = "/exp/sbnd/data/users/munjung/plots/numucc1p0pi"
today_str = datetime.now().strftime("%Y%m%d")
save_fig_dir = path.join(save_fig_base_dir, "systematics-genie-{}".format(today_str))

if save_fig:
    if not path.exists(save_fig_dir):
        makedirs(save_fig_dir)
    print("saving plots in ", save_fig_dir)

In [ ]:
file_dir = "/exp/sbnd/data/users/munjung/xsec/2025Spring_v10_06_00_09"

## -- MC 
mc_file = path.join(file_dir, "MC", "BNB_cosmics", "aa_sel_mup-geniewgts.df")
mc_split_df = pd.read_hdf(mc_file, key="split")
mc_n_split = get_n_split(mc_file)
print("mc_n_split: %d" %(mc_n_split))
print_keys(mc_file)

n_max_concat = 3
mc_keys2load = ['hdr', "mcnu", 'evt'] 
mc_dfs = load_dfs(mc_file, mc_keys2load, n_max_concat=n_max_concat)
mc_hdr_df = mc_dfs['hdr']
mc_nu_df = mc_dfs['mcnu']
mc_evt_df = mc_dfs['evt']

In [ ]:
## total pot
mc_tot_pot = mc_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))

# target_pot = 1e20
target_pot = mc_tot_pot
mc_pot_scale = target_pot / mc_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))

mc_evt_df["pot_weight"] = mc_pot_scale * np.ones(len(mc_evt_df))
mc_nu_df["pot_weight"] = mc_pot_scale * np.ones(len(mc_nu_df))

In [ ]:
# add multiindex column index "mc" so that branch names match evt_df
# TODO: fix so I don't have to do this
mc_nu_df.columns = pd.MultiIndex.from_tuples([tuple(["mc"] + list(c)) for c in mc_nu_df.columns])     # match # of column levels

In [ ]:
print("==== breakdown of selected events ====")
mc_evt_df.loc[:,'nuint_categ'] = get_int_category(mc_evt_df)
mc_evt_df.loc[:,'genie_categ'] = get_genie_category(mc_evt_df)
print(mc_evt_df.nuint_categ.value_counts())
print(mc_evt_df.genie_categ.value_counts())

print("==== breakdown of all MC events ====")
mc_nu_df.loc[:,'nuint_categ'] = get_int_category(mc_nu_df)
mc_nu_df.loc[:,'genie_categ'] = get_genie_category(mc_nu_df)
print(mc_nu_df.nuint_categ.value_counts())
print(mc_nu_df.genie_categ.value_counts())

In [ ]:
syst_name = ("mc", "GENIE")
mc_evt_df[syst_name]

In [ ]:
var_config = VariableConfig.muon_momentum()

In [ ]:
signal_hists(evtdf=mc_evt_df, 
             nudf=mc_nu_df, 
             var_config=var_config, 
             save_fig=save_fig, 
             save_name=None)

In [ ]:
# Note: these are background subtracted
cov_type = "xsec"
univ_events, cv_events = get_genie_univs(cov_type, mc_evt_df, mc_nu_df, var_config, syst_name)
ret_xsec = get_covariance_matrix(univ_events, cv_events)
plot_univ_hists(univ_events, cv_events, syst_name, var_config)

cov_type = "rate"
univ_events, cv_events = get_genie_univs(cov_type, mc_evt_df, mc_nu_df, var_config, syst_name)
ret_rate = get_covariance_matrix(univ_events, cv_events)
plot_univ_hists(univ_events, cv_events, syst_name, var_config)

In [ ]:
matrix_type = "cov"
save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_name, matrix_type)
title = "{} {}".format(syst_name, matrix_type)
plot_heatmap(ret_xsec[matrix_type], var_config, 
             title=title, 
             save_fig=save_fig, save_fig_name=save_fig_name)
plot_heatmap(ret_rate[matrix_type], var_config, 
             title=title, 
             save_fig=save_fig, save_fig_name=save_fig_name)

In [ ]:
frac_unc = np.sqrt(np.diag(ret_xsec["cov_frac"]))
plot_frac_unc(frac_unc, var_config)

frac_unc = np.sqrt(np.diag(ret_rate["cov_frac"]))
plot_frac_unc(frac_unc, var_config)

In [ ]:
if 'syst_dict' in locals():
    syst_dict[var_config.var_save_name] = ret_xsec["cov_frac"]
    syst_dict[var_config.var_save_name+"_rate"] = ret_rate["cov_frac"]

else:
    syst_dict = {}
    syst_dict[var_config.var_save_name] = ret_xsec["cov_frac"]
    syst_dict[var_config.var_save_name+"_rate"] = ret_rate["cov_frac"]

# save syst_dict as an npz file in the directory where dfs were loaded from
if save_result:
    print("saving syst_dict as npz in %s" % (file_dir))
    np.savez(file_dir + "/genie_syst_dict.npz", **syst_dict)

# sanity check: load saved file and check it agrees with the local one for all keys
loaded = np.load(file_dir + "/genie_syst_dict.npz")
for key in syst_dict.keys():
    arr_local = syst_dict[key]
    arr_loaded = loaded[key]
    assert np.allclose(arr_local, arr_loaded), f"Mismatch for key: {key}"
print("All keys in syst_dict match the saved npz file.")